# NB62: Smart City Traffic

Kafka -> Spark -> Redis/MinIO

## 1. Environment Setup

Installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and Python libraries (PySpark, Kafka-Python, Redis, Mongo, ES, Cassandra, MinIO).

In [ ]:
# Install Dependencies (Java 8, Spark 3.5.0, Kafka 3.6.1)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip uninstall -y numpy
!pip install -q "numpy<2.0.0"
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

Starts background services needed for this pipeline:
- **Kafka** (Zookeeper + Broker)
- **Redis**

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Redis
!apt-get install redis-server -qq > /dev/null
!service redis-server start

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(6379) # Redis


In [ ]:
# Start MinIO on Custom Port 9010 to avoid conflicts
!wget -q https://dl.min.io/server/minio/release/linux-amd64/minio
!chmod +x minio
!mkdir -p /content/minio_data_nb62
!MINIO_ROOT_USER=minioadmin MINIO_ROOT_PASSWORD=minioadmin ./minio server /content/minio_data_nb62 --address ":9010" --console-address ":9011" &> minio_9010.log &

# Wait for MinIO 9010
import time, socket, os
print('Waiting for MinIO on 9010...')
start = time.time()
while True:
    try:
        with socket.create_connection(('localhost', 9010), timeout=1): break
    except (OSError, ConnectionRefusedError):
        if time.time() - start > 120:
             if os.path.exists('minio_9010.log'): print(open('minio_9010.log').read())
             raise Exception('MinIO 9010 Failed')
        time.sleep(1)
print('MinIO 9010 Ready!')

## 3. Create Kafka Topic

Creates a topic named `input-topic`.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer (Traffic Sensors)

Simulates speed data from sensors.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Traffic Simulator...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 500 sensor readings...")
for _ in range(500):
    data = {'sensor_id': f's{random.randint(1,5)}', 'speed': random.randint(0, 120)}
    producer.send('input-topic', json.dumps(data).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Traffic Control Pipeline

1. Calcs Avg Speed per batch.
2. Updates Traffic Light status in Redis (Green/Red).
3. Archives raw batch to MinIO (Port 9010).

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
import redis, json
from minio import Minio

# Init MinIO (Port 9010)
m_client = Minio("127.0.0.1:9010", access_key="minioadmin", secret_key="minioadmin", secure=False)
if not m_client.bucket_exists("traffic-archive"): m_client.make_bucket("traffic-archive")

spark = SparkSession.builder.appName("SmartCity").getOrCreate()

def process_batch(df, epoch_id):
    data = [json.loads(r.value) for r in df.collect()]
    if not data: return
    
    # Agg Logic
    avg_speed = sum(d['speed'] for d in data) / len(data)
    
    # Redis Update
    r = redis.Redis()
    status = "GREEN" if avg_speed > 40 else "RED"
    r.set("traffic:status", status)
    
    # MinIO Archive
    import io
    content = json.dumps(data).encode('utf-8')
    m_client.put_object("traffic-archive", f"batch_{epoch_id}.json", io.BytesIO(content), len(content))
    print(f"Batch {epoch_id}: Avg Speed {avg_speed:.1f} -> {status}. Archived to MinIO.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Check Redis Status and MinIO Archives.

In [ ]:
import redis
from minio import Minio
r = redis.Redis()
print(f"Traffic Status: {r.get('traffic:status')}")

m = Minio("127.0.0.1:9010", access_key="minioadmin", secret_key="minioadmin", secure=False)
print(f"Archives: {len(list(m.list_objects('traffic-archive')))} files.")